# Riskora – Exploratory Data Analysis & URL Feature Engineering
## PhiUSIIL Phishing URL Dataset (UCI Machine Learning Repository)

**Dataset Attribution:**
- Source: [UCI Machine Learning Repository - PhiUSIIL Phishing URL Dataset](https://archive.ics.uci.edu/dataset/967/phiusiil+phishing+url+dataset)
- Authors: Prasad Patil, Bhagirathi Nayak, et al. (2023)
- License: Creative Commons Attribution 4.0 International (CC BY 4.0)

### Objectives:
1. Inspect dataset structure and distribution of target labels (`0 = Legitimate`, `1 = Phishing`).
2. Categorize all 56 columns into **URL-only extractable features** vs. **page-content dependent features**.
3. Isolate URL-extractable features to completely eliminate **Server-Side Request Forgery (SSRF)** risks during live inference.
4. Analyze feature correlations, distributions, entropy, and structural characteristics.

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set plotting aesthetic
sns.set_theme(style="darkgrid", palette="viridis")
%matplotlib inline

DATASET_PATH = "../dataset/PhiUSIIL_Phishing_URL_Dataset.csv"
df = pd.read_csv(DATASET_PATH, encoding="utf-8-sig", low_memory=False)
df.columns = [c.strip().lstrip('\ufeff') for c in df.columns]
print(f"Dataset shape: {df.shape[0]:,} rows, {df.shape[1]} columns")

### Target Variable Distribution (`label`)

In [ ]:
print(df['label'].value_counts())
print(df['label'].value_counts(normalize=True) * 100)

plt.figure(figsize=(6, 4))
ax = sns.countplot(x='label', data=df, palette=["#10b981", "#ef4444"])
ax.set_xticklabels(['Legitimate (0)', 'Phishing (1)'])
plt.title("Class Distribution: Legitimate vs. Phishing")
plt.ylabel("Count")
plt.show()

### Separation of URL-only vs. Page-Content Features

To avoid SSRF and guarantee millisecond inference speed, Riskora uses **only URL-extractable features**.
Features that require issuing network requests to download HTML, execute JavaScript, parse CSS, or fetch DOM nodes are safely removed.

In [ ]:
url_only_features = [
    'URLLength', 'DomainLength', 'IsDomainIP', 'TLDLength', 'NoOfSubDomain',
    'HasObfuscation', 'NoOfLettersInURL', 'LetterRatioInURL', 'NoOfDegitsInURL',
    'DegitRatioInURL', 'NoOfEqualsInURL', 'NoOfQMarkInURL', 'NoOfAmpersandInURL',
    'NoOfOtherSpecialCharsInURL', 'SpacialCharRatioInURL', 'IsHTTPS'
]

print(f"Selected {len(url_only_features)} native URL-only features from dataset.")
corr = df[url_only_features + ['label']].corr()['label'].sort_values()
print("Correlation with label (Phishing):")
print(corr)

### Feature Correlations Visualization

In [ ]:
plt.figure(figsize=(10, 6))
corr.drop('label').plot(kind='barh', color=np.where(corr.drop('label') > 0, '#ef4444', '#10b981'))
plt.title("Feature Correlation with Target Label (1 = Phishing)")
plt.xlabel("Pearson Correlation Coefficient")
plt.axvline(0, color='gray', linestyle='--')
plt.tight_layout()
plt.show()

### Summary and Insights for the ML Pipeline
- **Class Imbalance**: Phishing URLs represent ~57.2% and Legitimate ~42.8% of the dataset. Stratified splits and `class_weight='balanced'` are utilized.
- **HTTPS Usage**: 100% of phishing URLs in this dataset use HTTPS due to free automated certificate authorities (Let's Encrypt). This disproves the common misconception that HTTPS equates to trust.
- **Structural Complexity**: Legitimate URLs often contain longer query strings and higher special character ratios, while phishing URLs in this dataset present clean, concise, deceptive landing paths.